# MBDPO to model-free RL: `gradient2noise` → TD3

This notebook trains the repository's PyTorch `NeuralStateFeedbackPolicy`
with `R2Trainer` and gradient-driven action noise, freezes the actor after
exactly 1,500 full-horizon updates, and transfers that exact deterministic
network into a PyTorch TD3 implementation. All real-environment evaluations
use one fixed seed set and one shared action projection path.

The default run is intentionally substantial: `instance_4.rddl` has a
2,000-step horizon, MBDPO performs 1,500 full-horizon updates, and TD3 collects
50,000 real environment steps. For development only, set
`MDPO2RL_SMOKE_TEST=1` before launching the kernel; the checked-in defaults
remain unchanged.


## 1. Configuration and reproducibility

All experiment seeds and main run lengths are defined once here. The optional
smoke flag changes only effective development lengths; a normal run always
uses the requested 1,500 MBDPO iterations and 50,000 TD3 environment steps.


In [22]:
from __future__ import annotations

import copy
import hashlib
import json
import os
import random
from collections import deque
from pathlib import Path
from typing import Any, Dict, Iterable, Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyRDDLGym
import torch
from IPython.display import display
from torch import nn

from StochasticPBBP.core.Logic import (
    FuzzyLogic,
    ProductTNorm,
    SigmoidComparison,
    SoftControlFlow,
    SoftRandomSampling,
    SoftRounding,
)
from StochasticPBBP.core.R2Trainer import R2Trainer
from StochasticPBBP.core.Rollout import TorchRollout
from StochasticPBBP.utils.Noise import AdditiveNoiseFactory
from StochasticPBBP.utils.Policies import NeuralStateFeedbackPolicy
from StochasticPBBP.utils.R2Noise import R2GradientAdditiveNoise

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "StochasticPBBP").is_dir():
    raise RuntimeError("Run mdpo2rl.ipynb with the StochasticPBBP repository root as the working directory.")

DOMAIN_NAME = "reservoir"
INSTANCE_NAME = "instance_3.rddl"
DOMAIN_PATH = PROJECT_ROOT / "StochasticPBBP" / "problems" / DOMAIN_NAME / "domain.rddl"
INSTANCE_PATH = PROJECT_ROOT / "StochasticPBBP" / "problems" / DOMAIN_NAME / INSTANCE_NAME

PYTHON_SEED = 101
NUMPY_SEED = 202
TORCH_SEED = 303
MBDPO_TRAIN_SEED = 404
RL_TRAIN_SEED = 505
EVALUATION_SEEDS = (1101, 1102, 1103, 1104, 1105)
NUM_EVALUATION_EPISODES = len(EVALUATION_SEEDS)

HIDDEN_SIZES = (12, 12)
MBDPO_LEARNING_RATE = 0.01
DEFAULT_MBDPO_ITERATIONS = 1_500
DEFAULT_RL_TOTAL_STEPS = 120_000

GRADIENT2NOISE_MAX_STD = 1
GRADIENT2NOISE_MIN_STD = 0
GRADIENT2NOISE_ALPHA = 2
GRADIENT2NOISE_SCALE = 1.0

SMOKE_TEST = os.environ.get("MDPO2RL_SMOKE_TEST", "0") == "1"
MBDPO_ITERATIONS = 1 if SMOKE_TEST else DEFAULT_MBDPO_ITERATIONS
RL_TOTAL_STEPS = 64 if SMOKE_TEST else DEFAULT_RL_TOTAL_STEPS
SMOKE_HORIZON_STEPS = 8

default_output_dir = PROJECT_ROOT / "outputs" / "mdpo2rl"
OUTPUT_DIR = Path(os.environ.get("MDPO2RL_OUTPUT_DIR", str(default_output_dir))).resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(PYTHON_SEED)
np.random.seed(NUMPY_SEED)
torch.manual_seed(TORCH_SEED)
torch.use_deterministic_algorithms(True, warn_only=True)

print(f"mode={'SMOKE' if SMOKE_TEST else 'FULL'}")
print(f"MBDPO iterations={MBDPO_ITERATIONS:,}; TD3 steps={RL_TOTAL_STEPS:,}")
print(f"output directory={OUTPUT_DIR}")


mode=FULL
MBDPO iterations=1,500; TD3 steps=120,000
output directory=/Users/yuvalaroosh/StochasticPBBP/outputs/mdpo2rl


## 2. RDDL environment, differentiable rollout, and templates

The horizon is read from `instance_4.rddl`. Normal execution uses it
unchanged. Smoke mode caps only the development rollout/evaluation length so
every stage can be exercised quickly.


In [23]:
def make_real_env():
    return pyRDDLGym.make(
        domain=str(DOMAIN_PATH),
        instance=str(INSTANCE_PATH),
        vectorized=True,
    )


eval_env = make_real_env()
RDDL_HORIZON = int(eval_env.model.horizon)
HORIZON = min(RDDL_HORIZON, SMOKE_HORIZON_STEPS) if SMOKE_TEST else RDDL_HORIZON
DISCOUNT = float(eval_env.discount)

template_rollout = TorchRollout(
    eval_env.model,
    horizon=HORIZON,
    key=torch.Generator().manual_seed(MBDPO_TRAIN_SEED),
)
_, observation_template, _ = template_rollout.reset()
action_template = template_rollout.noop_actions

print(f"RDDL horizon={RDDL_HORIZON:,}; effective horizon={HORIZON:,}; discount={DISCOUNT}")
print("observation template:", {k: tuple(v.shape) for k, v in observation_template.items()})
print("action template:", {k: tuple(v.shape) for k, v in action_template.items()})


RDDL horizon=20; effective horizon=20; discount=1.0
observation template: {'rlevel': (10,)}
action template: {'release': (10,)}


/Users/yuvalaroosh/miniconda3/envs/drp/lib/python3.10/site-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/Users/yuvalaroosh/miniconda3/envs/drp/lib/python3.10/site-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


## 3. MBDPO components and a single verified action representation

The fuzzy-logic settings match `R2N_run.py`. `gradient2noise` remains the
same object for the entire `train_trajectory` call, so its time-indexed
profile persists across all iterations. The analysis rollout uses exactly
zero additive noise.

`NeuralStateFeedbackPolicy` currently does not numerically enforce its action
bounds. The helpers below therefore derive bounds from `env.action_space`,
lock observation/action ordering to the policy specs, and perform the only
projection used for every real-environment evaluation and TD3 interaction.


In [24]:
policy = NeuralStateFeedbackPolicy(
    observation_template=observation_template,
    action_template=action_template,
    action_space=eval_env.action_space,
    hidden_sizes=HIDDEN_SIZES,
    seed=MBDPO_TRAIN_SEED,
)

initial_untrained_policy = copy.deepcopy(policy)
initial_untrained_policy.eval()
for parameter in initial_untrained_policy.parameters():
    parameter.requires_grad_(False)

fuzzy_logic = FuzzyLogic(
    tnorm=ProductTNorm(),
    comparison=SigmoidComparison(weight=100.0),
    rounding=SoftRounding(weight=100.0),
    control=SoftControlFlow(weight=100.0),
    sampling=SoftRandomSampling(
        poisson_max_bins=100,
        binomial_max_bins=100,
        bernoulli_gumbel_softmax=False,
    ),
)

gradient2noise = R2GradientAdditiveNoise(
    scale=GRADIENT2NOISE_SCALE,
    eps=1e-6,
    min_std=GRADIENT2NOISE_MIN_STD,
    max_std=GRADIENT2NOISE_MAX_STD,
    alpha=GRADIENT2NOISE_ALPHA,
    norm_scope="global",
    step_score_aggregate="mean",
    normalization_quantile=0.95,
)
gradient2noise.g.manual_seed(MBDPO_TRAIN_SEED)

zero_analysis_noise = AdditiveNoiseFactory.create(
    noise_type="constant",
    std=0.0,
    source=template_rollout,
)

trainer = R2Trainer(
    model=eval_env.model,
    action_space=eval_env.action_space,
    policy=policy,
    horizon=HORIZON,
    lr=MBDPO_LEARNING_RATE,
    hidden_sizes=HIDDEN_SIZES,
    additive_noise=gradient2noise,
    analysis_additive_noise=zero_analysis_noise,
    logic=fuzzy_logic,
    seed=MBDPO_TRAIN_SEED,
    batch_size=HORIZON,
    batch_num=1,
)


In [25]:
OBSERVATION_SPECS = tuple(copy.deepcopy(policy.observation_specs))
ACTION_SPECS = tuple(copy.deepcopy(policy.action_specs))
OBSERVATION_ORDER = tuple(spec["name"] for spec in OBSERVATION_SPECS)
ACTION_ORDER = tuple(spec["name"] for spec in ACTION_SPECS)

assert len(OBSERVATION_ORDER) == len(set(OBSERVATION_ORDER))
assert len(ACTION_ORDER) == len(set(ACTION_ORDER))
assert ACTION_ORDER == tuple(eval_env.action_space.spaces.keys())

ACTION_LOW_NP = np.concatenate([
    np.asarray(eval_env.action_space.spaces[name].low, dtype=np.float32).reshape(-1)
    for name in ACTION_ORDER
])
ACTION_HIGH_NP = np.concatenate([
    np.asarray(eval_env.action_space.spaces[name].high, dtype=np.float32).reshape(-1)
    for name in ACTION_ORDER
])
ACTION_LOW = torch.as_tensor(ACTION_LOW_NP, dtype=torch.float32)
ACTION_HIGH = torch.as_tensor(ACTION_HIGH_NP, dtype=torch.float32)
ACTION_RANGE = ACTION_HIGH - ACTION_LOW
STATE_DIM = sum(int(spec["numel"]) for spec in OBSERVATION_SPECS)
ACTION_DIM = sum(int(spec["numel"]) for spec in ACTION_SPECS)

assert ACTION_LOW.shape == ACTION_HIGH.shape == (ACTION_DIM,)
assert torch.all(torch.isfinite(ACTION_LOW))
assert torch.all(torch.isfinite(ACTION_HIGH))
assert torch.all(ACTION_HIGH > ACTION_LOW)


def _strip_leading_singletons(tensor: torch.Tensor, target_shape: tuple[int, ...]) -> torch.Tensor:
    while tuple(tensor.shape) != target_shape and tensor.ndim > 0 and tensor.shape[0] == 1:
        tensor = tensor.reshape(tensor.shape[1:])
    return tensor


def flatten_observation(observation: Dict[str, Any]) -> torch.Tensor:
    parts = []
    for spec in OBSERVATION_SPECS:
        name = spec["name"]
        if name not in observation:
            raise KeyError(f"Missing observation fluent <{name}>.")
        tensor = torch.as_tensor(observation[name], dtype=torch.float32)
        tensor = _strip_leading_singletons(tensor, tuple(spec["shape"]))
        if tuple(tensor.shape) != tuple(spec["shape"]):
            raise ValueError(
                f"Observation <{name}> must have shape {spec['shape']}, got {tuple(tensor.shape)}."
            )
        parts.append(tensor.reshape(-1))
    flat = torch.cat(parts)
    assert flat.shape == (STATE_DIM,)
    return flat


def flatten_action(action: Dict[str, Any]) -> torch.Tensor:
    parts = []
    for spec in ACTION_SPECS:
        name = spec["name"]
        if name not in action:
            raise KeyError(f"Missing action fluent <{name}>.")
        tensor = torch.as_tensor(action[name], dtype=torch.float32)
        if tuple(tensor.shape) != tuple(spec["shape"]):
            raise ValueError(f"Action <{name}> must have shape {spec['shape']}.")
        parts.append(tensor.reshape(-1))
    flat = torch.cat(parts)
    assert flat.shape == (ACTION_DIM,)
    return flat


def unflatten_action(flat_action: torch.Tensor | np.ndarray) -> Dict[str, np.ndarray]:
    flat = torch.as_tensor(flat_action, dtype=torch.float32).reshape(-1)
    if flat.shape != (ACTION_DIM,):
        raise ValueError(f"Flat action must have shape {(ACTION_DIM,)}, got {tuple(flat.shape)}.")
    result: Dict[str, np.ndarray] = {}
    start = 0
    for spec in ACTION_SPECS:
        end = start + int(spec["numel"])
        name = spec["name"]
        space = eval_env.action_space.spaces[name]
        result[name] = (
            flat[start:end]
            .reshape(tuple(spec["shape"]))
            .detach()
            .cpu()
            .numpy()
            .astype(space.dtype, copy=False)
        )
        start = end
    assert start == ACTION_DIM
    return result


def project_action(flat_action: torch.Tensor) -> torch.Tensor:
    low = ACTION_LOW.to(device=flat_action.device, dtype=flat_action.dtype)
    high = ACTION_HIGH.to(device=flat_action.device, dtype=flat_action.dtype)
    return torch.maximum(torch.minimum(flat_action, high), low)


def assert_action_in_bounds(flat_action: torch.Tensor, atol: float = 1e-6) -> None:
    flat = flat_action.detach().cpu().reshape(-1)
    if not (torch.all(flat >= ACTION_LOW - atol) and torch.all(flat <= ACTION_HIGH + atol)):
        raise AssertionError("An action passed to env.step was outside env.action_space bounds.")


round_trip_probe = (ACTION_LOW + ACTION_HIGH) / 2.0
torch.testing.assert_close(flatten_action(unflatten_action(round_trip_probe)), round_trip_probe)
print("fixed observation order:", OBSERVATION_ORDER)
print("fixed action order:", ACTION_ORDER)
print(f"state_dim={STATE_DIM}, action_dim={ACTION_DIM}")


fixed observation order: ('rlevel',)
fixed action order: ('release',)
state_dim=10, action_dim=10


In [26]:
EVALUATION_PROTOCOL = {
    "seeds": EVALUATION_SEEDS,
    "episodes": NUM_EVALUATION_EPISODES,
    "horizon": HORIZON,
    "discount": DISCOUNT,
    "projection": "componentwise env.action_space bounds",
    "observation_order": OBSERVATION_ORDER,
    "action_order": ACTION_ORDER,
}


@torch.no_grad()
def evaluate_actor(actor_network: nn.Module, label: str) -> Dict[str, Any]:
    was_training = actor_network.training
    actor_network.eval()
    returns = []
    violating_action_vectors = 0
    raw_action_vectors = 0
    violating_action_components = 0
    raw_action_components = 0

    try:
        for seed in EVALUATION_SEEDS:
            observation, _ = eval_env.reset(seed=int(seed))
            total_return = 0.0
            cumulative_discount = 1.0

            for _ in range(HORIZON):
                state = flatten_observation(observation)
                raw_action = actor_network(state)
                if raw_action.shape != (ACTION_DIM,):
                    raise AssertionError(
                        f"Actor output must have shape {(ACTION_DIM,)}, got {tuple(raw_action.shape)}."
                    )

                low = ACTION_LOW.to(raw_action)
                high = ACTION_HIGH.to(raw_action)
                violation_mask = (raw_action < low) | (raw_action > high)
                violating_action_vectors += int(torch.any(violation_mask).item())
                raw_action_vectors += 1
                violating_action_components += int(torch.count_nonzero(violation_mask).item())
                raw_action_components += int(raw_action.numel())

                executed_action = project_action(raw_action)
                assert_action_in_bounds(executed_action)
                env_action = unflatten_action(executed_action)
                observation, reward, terminated, truncated, _ = eval_env.step(env_action)
                total_return += cumulative_discount * float(reward)
                cumulative_discount *= DISCOUNT
                if bool(terminated) or bool(truncated):
                    break

            returns.append(total_return)
    finally:
        actor_network.train(was_training)

    values = np.asarray(returns, dtype=np.float64)
    return {
        "policy": label,
        "mean": float(values.mean()),
        "std": float(values.std()),
        "median": float(np.median(values)),
        "min": float(values.min()),
        "max": float(values.max()),
        "raw_action_violation_pct": 100.0 * violating_action_vectors / max(raw_action_vectors, 1),
        "raw_action_component_violation_pct": 100.0 * violating_action_components / max(raw_action_components, 1),
        "returns": tuple(float(value) for value in values),
        "evaluation_protocol": copy.deepcopy(EVALUATION_PROTOCOL),
    }


initial_evaluation = evaluate_actor(
    initial_untrained_policy.network,
    "Before MBDPO training",
)
display(pd.DataFrame([{k: v for k, v in initial_evaluation.items() if k not in {"returns", "evaluation_protocol"}}]))


,policy,mean,std,median,min,max,raw_action_violation_pct,raw_action_component_violation_pct
0,Before MBDPO training,-67421.970969,0.0,-67421.970969,-67421.970969,-67421.970969,100.0,30.0


## 4. Continuous MBDPO training with `gradient2noise`

This is one uninterrupted call. Each outer iteration runs one full-horizon
optimizer update, then a zero-noise analysis rollout that refreshes the
profile used by the next iteration.


In [27]:
mbdpo_history, trained_policy = trainer.train_trajectory(
    iterations=MBDPO_ITERATIONS,
    print_every=1 if SMOKE_TEST else 25,
    batch_size=HORIZON,
    batch_num=1,
    additive_noise=gradient2noise,
    analysis_additive_noise=zero_analysis_noise,
)

assert trained_policy is policy
assert len(mbdpo_history) == MBDPO_ITERATIONS
assert int(mbdpo_history[-1]["iteration"]) == MBDPO_ITERATIONS
assert all(int(row["update_steps"]) == HORIZON for row in mbdpo_history)
if not SMOKE_TEST:
    assert MBDPO_ITERATIONS == DEFAULT_MBDPO_ITERATIONS == 1_500
    assert len(mbdpo_history) == 1_500

mbdpo_history_df = pd.DataFrame(mbdpo_history)
display(mbdpo_history_df[[
    "iteration",
    "update_return",
    "analysis_return",
    "update_loss",
    "sigma_mean",
    "sigma_min",
    "sigma_max",
]].tail())


iter=   1 update_return=-67421.9766 update_loss=67421.9766 analysis_return=-62174.3789 steps=20
iter=  25 update_return=-33655.4102 update_loss=33655.4102 analysis_return=-33445.8164 steps=20
iter=  50 update_return=-33847.8203 update_loss=33847.8203 analysis_return=-33434.3242 steps=20
iter=  75 update_return=-31183.6250 update_loss=31183.6250 analysis_return=-31134.6719 steps=20
iter= 100 update_return=-29967.0840 update_loss=29967.0840 analysis_return=-29865.4844 steps=20
iter= 125 update_return=-29394.2344 update_loss=29394.2344 analysis_return=-29385.6289 steps=20
iter= 150 update_return=-29554.0684 update_loss=29554.0684 analysis_return=-29565.5859 steps=20
iter= 175 update_return=-30970.7617 update_loss=30970.7617 analysis_return=-30860.2754 steps=20
iter= 200 update_return=-29696.2422 update_loss=29696.2422 analysis_return=-29630.6484 steps=20
iter= 225 update_return=-29113.7559 update_loss=29113.7559 analysis_return=-29092.5391 steps=20
iter= 250 update_return=-29149.1230 upda

,iteration,update_return,analysis_return,update_loss,sigma_mean,sigma_min,sigma_max
1495,1496.0,-27848.154297,-27776.392578,27848.154297,0.470782,0.0,0.891252
1496,1497.0,-27769.513672,-27771.437500,27769.513672,0.470782,0.0,0.891252
1497,1498.0,-27822.095703,-27762.857422,27822.095703,0.470630,0.0,0.891918
1498,1499.0,-27692.898438,-27754.763672,27692.898438,0.464778,0.0,0.882528
1499,1500.0,-27939.451172,-27748.386719,27939.451172,0.436827,0.0,0.845646


In [28]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
axes[0].plot(mbdpo_history_df["iteration"], mbdpo_history_df["update_return"], label="update return", alpha=0.75)
axes[0].plot(mbdpo_history_df["iteration"], mbdpo_history_df["analysis_return"], label="analysis return")
axes[0].set(title="MBDPO returns", xlabel="iteration", ylabel="full-horizon return")
axes[0].legend()

axes[1].plot(mbdpo_history_df["iteration"], mbdpo_history_df["update_loss"])
axes[1].set(title="MBDPO update loss", xlabel="iteration", ylabel="loss")

axes[2].plot(mbdpo_history_df["iteration"], mbdpo_history_df["sigma_mean"], label="mean")
axes[2].plot(mbdpo_history_df["iteration"], mbdpo_history_df["sigma_min"], label="min")
axes[2].plot(mbdpo_history_df["iteration"], mbdpo_history_df["sigma_max"], label="max")
axes[2].set(title="gradient2noise profile", xlabel="iteration", ylabel="sigma")
axes[2].legend()
fig.tight_layout()
plt.show()


/var/folders/3m/fcqmj73x6_755l29dw9gpsmw0000gn/T/ipykernel_33622/1607582651.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Stop at iteration 1,500, freeze, checkpoint, and evaluate

The trained policy is frozen immediately after training. The baseline copy
remains immutable throughout TD3, and its checkpoint file is hashed before
and after model-free training.


In [29]:
trained_policy.eval()
for parameter in trained_policy.parameters():
    parameter.requires_grad_(False)
assert all(not parameter.requires_grad for parameter in trained_policy.parameters())

mbdpo_baseline = copy.deepcopy(trained_policy)
mbdpo_baseline.eval()
for parameter in mbdpo_baseline.parameters():
    parameter.requires_grad_(False)

mbdpo_baseline_state = {
    name: tensor.detach().cpu().clone()
    for name, tensor in mbdpo_baseline.state_dict().items()
}

mbdpo_checkpoint_path = OUTPUT_DIR / "mdpo_actor_after_1500.pt"
torch.save(
    {
        "policy_class": type(mbdpo_baseline).__name__,
        "policy_state_dict": mbdpo_baseline.state_dict(),
        "network_state_dict": mbdpo_baseline.network.state_dict(),
        "hidden_sizes": HIDDEN_SIZES,
        "completed_iterations": MBDPO_ITERATIONS,
        "requested_full_iterations": DEFAULT_MBDPO_ITERATIONS,
        "smoke_test": SMOKE_TEST,
        "rddl_horizon": RDDL_HORIZON,
        "effective_horizon": HORIZON,
        "observation_order": OBSERVATION_ORDER,
        "action_order": ACTION_ORDER,
    },
    mbdpo_checkpoint_path,
)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


mbdpo_checkpoint_hash_before_td3 = sha256_file(mbdpo_checkpoint_path)
mbdpo_evaluation = evaluate_actor(
    mbdpo_baseline.network,
    "Frozen MBDPO after 1,500 iterations",
)
display(pd.DataFrame([{k: v for k, v in mbdpo_evaluation.items() if k not in {"returns", "evaluation_protocol"}}]))
print("MBDPO checkpoint:", mbdpo_checkpoint_path)


,policy,mean,std,median,min,max,raw_action_violation_pct,raw_action_component_violation_pct
0,"Frozen MBDPO after 1,500 iterations",-27748.383651,3.637979e-12,-27748.383651,-27748.383651,-27748.383651,100.0,30.0


MBDPO checkpoint: /Users/yuvalaroosh/StochasticPBBP/outputs/mdpo2rl/mdpo_actor_after_1500.pt


## 6. Exact actor transfer to TD3

TD3 fits this handoff because the pretrained policy is deterministic and the
environment has continuous bounded actions. The online actor below is a deep
copy of the frozen MBDPO policy; only the copied parameters are re-enabled for
gradients. No layer is reinitialized.


In [30]:
td3_policy = copy.deepcopy(mbdpo_baseline)
for parameter in td3_policy.parameters():
    parameter.requires_grad_(True)
td3_policy.train()
actor = td3_policy.network

actor_target = copy.deepcopy(actor)
actor_target.eval()
for parameter in actor_target.parameters():
    parameter.requires_grad_(False)

handoff_observation, _ = eval_env.reset(seed=EVALUATION_SEEDS[0])
handoff_state = flatten_observation(handoff_observation)
with torch.no_grad():
    baseline_handoff_action = mbdpo_baseline.network(handoff_state)
    td3_handoff_action = actor(handoff_state)
handoff_max_abs_difference = float(
    torch.max(torch.abs(td3_handoff_action - baseline_handoff_action)).item()
)
assert handoff_max_abs_difference <= 1e-8
print(f"handoff max |Δ action| = {handoff_max_abs_difference:.3e}")

actor_state_at_handoff = {
    name: tensor.detach().cpu().clone()
    for name, tensor in actor.state_dict().items()
}
td3_before_evaluation = evaluate_actor(actor, "TD3 actor before any RL update")
np.testing.assert_allclose(
    td3_before_evaluation["returns"],
    mbdpo_evaluation["returns"],
    rtol=0.0,
    atol=1e-8,
)


handoff max |Δ action| = 0.000e+00


## 7. TD3 configuration

“Warm-up” means the number of pretrained-actor transitions collected before
gradient updates begin. It does **not** replace the actor with random actions:
collection starts from the pretrained actor plus range-scaled exploration
noise.


In [31]:
TD3_CONFIG = {
    "total_environment_steps": RL_TOTAL_STEPS,
    "replay_buffer_capacity": 512 if SMOKE_TEST else 100_000,
    "warmup_steps_before_updates": 16 if SMOKE_TEST else 5_000,
    "batch_size": 16 if SMOKE_TEST else 256,
    "actor_learning_rate": 3e-4,
    "critic_learning_rate": 1e-3,
    "gamma": DISCOUNT,
    "tau": 0.005,
    "policy_delay": 2,
    "exploration_noise_scale": 0.10,
    "target_policy_noise_scale": 0.20,
    "target_policy_noise_clip": 0.50,
    "evaluation_frequency": 32 if SMOKE_TEST else 5_000,
    "critic_hidden_sizes": (256, 256),
}
display(pd.Series(TD3_CONFIG, name="value").to_frame())


,value
total_environment_steps,120000
replay_buffer_capacity,100000
warmup_steps_before_updates,5000
batch_size,256
actor_learning_rate,0.0003
critic_learning_rate,0.001
gamma,1.0
tau,0.005
policy_delay,2
exploration_noise_scale,0.1


## 8. PyTorch TD3 implementation

The replay buffer stores the projected action actually executed by the real
`pyRDDLGym` environment. Episode reset uses `terminated or truncated`, while
the Bellman bootstrap mask uses only `terminated`, following Gymnasium's
distinction between an MDP terminal state and a time-limit truncation.


In [32]:
class ReplayBuffer:
    def __init__(self, capacity: int, state_dim: int, action_dim: int):
        self.capacity = int(capacity)
        self.states = np.empty((capacity, state_dim), dtype=np.float32)
        self.actions = np.empty((capacity, action_dim), dtype=np.float32)
        self.rewards = np.empty((capacity, 1), dtype=np.float32)
        self.next_states = np.empty((capacity, state_dim), dtype=np.float32)
        self.terminated = np.empty((capacity, 1), dtype=np.float32)
        self.truncated = np.empty((capacity, 1), dtype=np.float32)
        self.position = 0
        self.size = 0

    def add(self, state, action, reward, next_state, terminated, truncated):
        index = self.position
        self.states[index] = np.asarray(state, dtype=np.float32)
        self.actions[index] = np.asarray(action, dtype=np.float32)
        self.rewards[index, 0] = float(reward)
        self.next_states[index] = np.asarray(next_state, dtype=np.float32)
        self.terminated[index, 0] = float(bool(terminated))
        self.truncated[index, 0] = float(bool(truncated))
        self.position = (self.position + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size: int, rng: np.random.Generator):
        indices = rng.integers(0, self.size, size=batch_size)
        return tuple(
            torch.as_tensor(array[indices], dtype=torch.float32)
            for array in (
                self.states,
                self.actions,
                self.rewards,
                self.next_states,
                self.terminated,
                self.truncated,
            )
        )

    def __len__(self):
        return self.size


class QNetwork(nn.Module):
    def __init__(self, state_dim: int, action_dim: int, hidden_sizes: Sequence[int]):
        super().__init__()
        layers = []
        input_dim = state_dim + action_dim
        for hidden_dim in hidden_sizes:
            layers.extend((nn.Linear(input_dim, hidden_dim), nn.ReLU()))
            input_dim = hidden_dim
        layers.append(nn.Linear(input_dim, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, state: torch.Tensor, action: torch.Tensor) -> torch.Tensor:
        return self.network(torch.cat((state, action), dim=-1))


def freeze_target(network: nn.Module) -> nn.Module:
    network.eval()
    for parameter in network.parameters():
        parameter.requires_grad_(False)
    return network


@torch.no_grad()
def soft_update(target: nn.Module, source: nn.Module, tau: float) -> None:
    for target_parameter, source_parameter in zip(target.parameters(), source.parameters()):
        target_parameter.mul_(1.0 - tau).add_(source_parameter, alpha=tau)


torch.manual_seed(RL_TRAIN_SEED)
critic1 = QNetwork(STATE_DIM, ACTION_DIM, TD3_CONFIG["critic_hidden_sizes"])
critic2 = QNetwork(STATE_DIM, ACTION_DIM, TD3_CONFIG["critic_hidden_sizes"])
critic1_target = freeze_target(copy.deepcopy(critic1))
critic2_target = freeze_target(copy.deepcopy(critic2))

actor_optimizer = torch.optim.Adam(actor.parameters(), lr=TD3_CONFIG["actor_learning_rate"])
critic1_optimizer = torch.optim.Adam(critic1.parameters(), lr=TD3_CONFIG["critic_learning_rate"])
critic2_optimizer = torch.optim.Adam(critic2.parameters(), lr=TD3_CONFIG["critic_learning_rate"])
replay_buffer = ReplayBuffer(
    TD3_CONFIG["replay_buffer_capacity"],
    STATE_DIM,
    ACTION_DIM,
)


In [33]:
train_env = make_real_env()
exploration_rng = np.random.default_rng(RL_TRAIN_SEED)
replay_rng = np.random.default_rng(RL_TRAIN_SEED + 1)
target_noise_generator = torch.Generator().manual_seed(RL_TRAIN_SEED + 2)

observation, _ = train_env.reset(seed=RL_TRAIN_SEED)
state = flatten_observation(observation)
episode_index = 0
gradient_steps = 0
actor_update_count = 0
executed_action_checks = 0
critic1_received_gradients = False
critic2_received_gradients = False

critic_loss_steps = []
critic1_losses = []
critic2_losses = []
actor_loss_steps = []
actor_losses = []
td3_evaluation_steps = [0]
td3_evaluation_means = [td3_before_evaluation["mean"]]

actor.train()
critic1.train()
critic2.train()

for environment_step in range(1, TD3_CONFIG["total_environment_steps"] + 1):
    with torch.no_grad():
        raw_action = actor(state)

    exploration_std = (
        TD3_CONFIG["exploration_noise_scale"] * ACTION_RANGE.numpy()
    )
    exploration_noise = torch.as_tensor(
        exploration_rng.normal(loc=0.0, scale=exploration_std),
        dtype=torch.float32,
    )
    executed_action = project_action(raw_action + exploration_noise)
    assert_action_in_bounds(executed_action)
    executed_action_checks += 1

    next_observation, reward, terminated, truncated, _ = train_env.step(
        unflatten_action(executed_action)
    )
    next_state = flatten_observation(next_observation)

    replay_buffer.add(
        state.numpy(),
        executed_action.numpy(),
        reward,
        next_state.numpy(),
        terminated,
        truncated,
    )

    state = next_state
    if bool(terminated) or bool(truncated):
        episode_index += 1
        observation, _ = train_env.reset(seed=RL_TRAIN_SEED + episode_index)
        state = flatten_observation(observation)

    ready_to_update = (
        environment_step >= TD3_CONFIG["warmup_steps_before_updates"]
        and len(replay_buffer) >= TD3_CONFIG["batch_size"]
    )
    if ready_to_update:
        (
            batch_states,
            batch_actions,
            batch_rewards,
            batch_next_states,
            batch_terminated,
            batch_truncated,
        ) = replay_buffer.sample(TD3_CONFIG["batch_size"], replay_rng)
        gradient_steps += 1

        with torch.no_grad():
            target_raw_actions = actor_target(batch_next_states)
            target_noise_std = (
                TD3_CONFIG["target_policy_noise_scale"] * ACTION_RANGE
            ).expand_as(target_raw_actions)
            target_noise_limit = (
                TD3_CONFIG["target_policy_noise_clip"] * ACTION_RANGE
            ).expand_as(target_raw_actions)
            target_noise = torch.normal(
                mean=torch.zeros_like(target_raw_actions),
                std=target_noise_std,
                generator=target_noise_generator,
            )
            target_noise = torch.maximum(
                torch.minimum(target_noise, target_noise_limit),
                -target_noise_limit,
            )
            target_actions = project_action(target_raw_actions + target_noise)
            target_q = torch.minimum(
                critic1_target(batch_next_states, target_actions),
                critic2_target(batch_next_states, target_actions),
            )
            not_terminal = 1.0 - batch_terminated
            bellman_target = batch_rewards + TD3_CONFIG["gamma"] * not_terminal * target_q

        critic1_prediction = critic1(batch_states, batch_actions)
        critic2_prediction = critic2(batch_states, batch_actions)
        critic1_loss = nn.functional.mse_loss(critic1_prediction, bellman_target)
        critic2_loss = nn.functional.mse_loss(critic2_prediction, bellman_target)

        critic1_optimizer.zero_grad(set_to_none=True)
        critic1_loss.backward()
        critic1_received_gradients |= any(
            parameter.grad is not None for parameter in critic1.parameters()
        )
        critic1_optimizer.step()

        critic2_optimizer.zero_grad(set_to_none=True)
        critic2_loss.backward()
        critic2_received_gradients |= any(
            parameter.grad is not None for parameter in critic2.parameters()
        )
        critic2_optimizer.step()

        critic_loss_steps.append(environment_step)
        critic1_losses.append(float(critic1_loss.detach()))
        critic2_losses.append(float(critic2_loss.detach()))

        if gradient_steps % TD3_CONFIG["policy_delay"] == 0:
            actor_optimizer.zero_grad(set_to_none=True)
            policy_actions = project_action(actor(batch_states))
            actor_loss = -critic1(batch_states, policy_actions).mean()
            actor_loss.backward()
            actor_optimizer.step()
            actor_update_count += 1

            actor_loss_steps.append(environment_step)
            actor_losses.append(float(actor_loss.detach()))

            soft_update(actor_target, actor, TD3_CONFIG["tau"])
            soft_update(critic1_target, critic1, TD3_CONFIG["tau"])
            soft_update(critic2_target, critic2, TD3_CONFIG["tau"])

    should_evaluate = (
        environment_step % TD3_CONFIG["evaluation_frequency"] == 0
        or environment_step == TD3_CONFIG["total_environment_steps"]
    )
    if should_evaluate:
        evaluation = evaluate_actor(actor, f"TD3 at step {environment_step}")
        td3_evaluation_steps.append(environment_step)
        td3_evaluation_means.append(evaluation["mean"])
        print(
            f"step={environment_step:7d} replay={len(replay_buffer):6d} "
            f"updates={gradient_steps:6d} eval_mean={evaluation['mean']:.4f}"
        )

train_env.close()

assert gradient_steps > 0
assert actor_update_count > 0
assert critic1_received_gradients and critic2_received_gradients
assert all(parameter.grad is None for parameter in actor_target.parameters())
assert all(parameter.grad is None for parameter in critic1_target.parameters())
assert all(parameter.grad is None for parameter in critic2_target.parameters())
assert executed_action_checks == RL_TOTAL_STEPS


/Users/yuvalaroosh/miniconda3/envs/drp/lib/python3.10/site-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/Users/yuvalaroosh/miniconda3/envs/drp/lib/python3.10/site-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


step=   5000 replay=  5000 updates=     1 eval_mean=-27748.3837
step=  10000 replay= 10000 updates=  5001 eval_mean=-39746.2667
step=  15000 replay= 15000 updates= 10001 eval_mean=-35348.8568
step=  20000 replay= 20000 updates= 15001 eval_mean=-33251.0627
step=  25000 replay= 25000 updates= 20001 eval_mean=-33109.8105
step=  30000 replay= 30000 updates= 25001 eval_mean=-33595.2960
step=  35000 replay= 35000 updates= 30001 eval_mean=-32515.8084
step=  40000 replay= 40000 updates= 35001 eval_mean=-32880.2066
step=  45000 replay= 45000 updates= 40001 eval_mean=-33045.5151
step=  50000 replay= 50000 updates= 45001 eval_mean=-28972.0238
step=  55000 replay= 55000 updates= 50001 eval_mean=-29446.9277
step=  60000 replay= 60000 updates= 55001 eval_mean=-35302.6552
step=  65000 replay= 65000 updates= 60001 eval_mean=-39906.5049
step=  70000 replay= 70000 updates= 65001 eval_mean=-39770.2137
step=  75000 replay= 75000 updates= 70001 eval_mean=-39953.6557
step=  80000 replay= 80000 updates= 7500

## 9. Final evaluation, checkpoints, and acceptance checks

The final actor is evaluated with the same seeds, horizon, discount, ordering,
and action projection as all earlier policies. The CSV therefore compares
like with like.


In [34]:
td3_after_evaluation = evaluate_actor(actor, "TD3 actor after RL fine-tuning")

actor_changed = any(
    not torch.equal(actor.state_dict()[name].detach().cpu(), tensor)
    for name, tensor in actor_state_at_handoff.items()
)
assert actor_changed

for name, tensor in mbdpo_baseline_state.items():
    torch.testing.assert_close(
        mbdpo_baseline.state_dict()[name].detach().cpu(),
        tensor,
        rtol=0.0,
        atol=0.0,
    )

mbdpo_checkpoint_hash_after_td3 = sha256_file(mbdpo_checkpoint_path)
assert mbdpo_checkpoint_hash_after_td3 == mbdpo_checkpoint_hash_before_td3

td3_actor_path = OUTPUT_DIR / "td3_actor_final.pt"
td3_checkpoint_path = OUTPUT_DIR / "td3_full_checkpoint.pt"
torch.save(
    {
        "network_state_dict": actor.state_dict(),
        "source": "deep copy of mdpo_actor_after_1500.pt, then TD3 fine-tuning",
        "observation_order": OBSERVATION_ORDER,
        "action_order": ACTION_ORDER,
    },
    td3_actor_path,
)
torch.save(
    {
        "actor": actor.state_dict(),
        "actor_target": actor_target.state_dict(),
        "critic1": critic1.state_dict(),
        "critic2": critic2.state_dict(),
        "critic1_target": critic1_target.state_dict(),
        "critic2_target": critic2_target.state_dict(),
        "actor_optimizer": actor_optimizer.state_dict(),
        "critic1_optimizer": critic1_optimizer.state_dict(),
        "critic2_optimizer": critic2_optimizer.state_dict(),
        "td3_config": copy.deepcopy(TD3_CONFIG),
        "experiment_config": {
            "domain": DOMAIN_NAME,
            "instance": INSTANCE_NAME,
            "rddl_horizon": RDDL_HORIZON,
            "effective_horizon": HORIZON,
            "evaluation_seeds": EVALUATION_SEEDS,
            "smoke_test": SMOKE_TEST,
        },
        "gradient_steps": gradient_steps,
        "actor_update_count": actor_update_count,
    },
    td3_checkpoint_path,
)

evaluation_rows = []
for result in (
    initial_evaluation,
    mbdpo_evaluation,
    td3_before_evaluation,
    td3_after_evaluation,
):
    evaluation_rows.append({
        key: value
        for key, value in result.items()
        if key not in {"returns", "evaluation_protocol"}
    })

evaluation_results_df = pd.DataFrame(evaluation_rows)
evaluation_csv_path = OUTPUT_DIR / "evaluation_results.csv"
evaluation_results_df.to_csv(evaluation_csv_path, index=False)
display(evaluation_results_df)

protocols = [
    result["evaluation_protocol"]
    for result in (
        initial_evaluation,
        mbdpo_evaluation,
        td3_before_evaluation,
        td3_after_evaluation,
    )
]
assert all(protocol == protocols[0] for protocol in protocols[1:])
assert protocols[0]["seeds"] == EVALUATION_SEEDS
assert protocols[0]["horizon"] == HORIZON


,policy,mean,std,median,min,max,raw_action_violation_pct,raw_action_component_violation_pct
0,Before MBDPO training,-67421.970969,0.000000e+00,-67421.970969,-67421.970969,-67421.970969,100.0,30.0
1,"Frozen MBDPO after 1,500 iterations",-27748.383651,3.637979e-12,-27748.383651,-27748.383651,-27748.383651,100.0,30.0
2,TD3 actor before any RL update,-27748.383651,3.637979e-12,-27748.383651,-27748.383651,-27748.383651,100.0,30.0
3,TD3 actor after RL fine-tuning,-50488.503813,0.000000e+00,-50488.503813,-50488.503813,-50488.503813,70.0,31.0


In [35]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].plot(td3_evaluation_steps, td3_evaluation_means, marker="o")
axes[0, 0].set(
    title="TD3 evaluation return",
    xlabel="real environment steps",
    ylabel="mean fixed-seed return",
)

axes[0, 1].plot(critic_loss_steps, critic1_losses, label="Q1")
axes[0, 1].plot(critic_loss_steps, critic2_losses, label="Q2", alpha=0.8)
axes[0, 1].set(title="Critic losses", xlabel="real environment steps", ylabel="MSE")
axes[0, 1].legend()

axes[1, 0].plot(actor_loss_steps, actor_losses)
axes[1, 0].set(title="Delayed actor loss", xlabel="real environment steps", ylabel="-Q1")

comparison_labels = ["Before TD3", "After TD3"]
comparison_means = [td3_before_evaluation["mean"], td3_after_evaluation["mean"]]
comparison_stds = [td3_before_evaluation["std"], td3_after_evaluation["std"]]
axes[1, 1].bar(comparison_labels, comparison_means, yerr=comparison_stds, capsize=5)
axes[1, 1].set(title="Fixed-seed performance before vs after TD3", ylabel="return")

fig.tight_layout()
plt.show()


/var/folders/3m/fcqmj73x6_755l29dw9gpsmw0000gn/T/ipykernel_33622/3952549596.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [36]:
if SMOKE_TEST:
    assert len(mbdpo_history) == MBDPO_ITERATIONS == 1
else:
    assert len(mbdpo_history) == MBDPO_ITERATIONS == 1_500

acceptance_checks = {
    "MBDPO requested iterations completed": len(mbdpo_history) == MBDPO_ITERATIONS,
    "full run stops exactly at 1,500": SMOKE_TEST or len(mbdpo_history) == 1_500,
    "MBDPO baseline remains frozen": all(
        not parameter.requires_grad for parameter in mbdpo_baseline.parameters()
    ),
    "MBDPO checkpoint unchanged during TD3": (
        mbdpo_checkpoint_hash_before_td3 == mbdpo_checkpoint_hash_after_td3
    ),
    "handoff actions identical": handoff_max_abs_difference <= 1e-8,
    "TD3 actor parameters changed": actor_changed,
    "both critics received gradients": (
        critic1_received_gradients and critic2_received_gradients
    ),
    "target networks received no direct gradients": all(
        parameter.grad is None
        for network in (actor_target, critic1_target, critic2_target)
        for parameter in network.parameters()
    ),
    "all collected actions were bounds-checked": executed_action_checks == RL_TOTAL_STEPS,
    "evaluation protocol identical": all(protocol == protocols[0] for protocol in protocols),
    "fixed flatten/unflatten ordering verified": (
        ACTION_ORDER == tuple(eval_env.action_space.spaces.keys())
    ),
}
assert all(acceptance_checks.values())
display(pd.Series(acceptance_checks, name="passed").to_frame())

print("Saved artifacts:")
for path in (
    mbdpo_checkpoint_path,
    td3_actor_path,
    td3_checkpoint_path,
    evaluation_csv_path,
):
    print(" -", path)

eval_env.close()


,passed
MBDPO requested iterations completed,True
"full run stops exactly at 1,500",True
MBDPO baseline remains frozen,True
MBDPO checkpoint unchanged during TD3,True
handoff actions identical,True
TD3 actor parameters changed,True
both critics received gradients,True
target networks received no direct gradients,True
all collected actions were bounds-checked,True
evaluation protocol identical,True


Saved artifacts:
 - /Users/yuvalaroosh/StochasticPBBP/outputs/mdpo2rl/mdpo_actor_after_1500.pt
 - /Users/yuvalaroosh/StochasticPBBP/outputs/mdpo2rl/td3_actor_final.pt
 - /Users/yuvalaroosh/StochasticPBBP/outputs/mdpo2rl/td3_full_checkpoint.pt
 - /Users/yuvalaroosh/StochasticPBBP/outputs/mdpo2rl/evaluation_results.csv


## Interpretation

Compare the identical-seed rows and the final bar chart to determine whether
real-environment TD3 interactions improved the transferred actor. Improvement
is not guaranteed for every finite sample: TD3 is stochastic, the reservoir
horizon is long, and hard action projection creates saturated regions. The
notebook reports rather than assumes the outcome.
